# BM25 Math — Understanding Sparse Retrieval

This notebook walks through the BM25 scoring formula step by step.
We'll compute scores by hand so you understand exactly what the algorithm does.

In [ ]:
import math

# Sample corpus — 3 short documents
corpus = [
    "the supplier in China has low reliability",
    "the warehouse in Rotterdam handles European shipments",
    "the supplier in Taiwan makes semiconductor chips",
]

# Tokenize (lowercase + split)
tokenized = [doc.lower().split() for doc in corpus]
for i, tokens in enumerate(tokenized):
    print(f"Doc {i}: {tokens}")

## Step 1: IDF — How rare is each word?

**IDF(t) = log((N - df(t) + 0.5) / (df(t) + 0.5))**

Where:
- N = total number of documents
- df(t) = number of documents containing term t

Rare words get HIGH IDF. Common words get LOW IDF.

In [ ]:
N = len(corpus)

# Count document frequency for each term
def doc_freq(term, tokenized_corpus):
    return sum(1 for doc in tokenized_corpus if term in doc)

def idf(term, tokenized_corpus):
    n = len(tokenized_corpus)
    df = doc_freq(term, tokenized_corpus)
    return math.log((n - df + 0.5) / (df + 0.5) + 1)

# Query terms
query = "supplier China"
query_terms = query.lower().split()

for term in query_terms:
    df = doc_freq(term, tokenized)
    idf_score = idf(term, tokenized)
    print(f"'{term}': appears in {df}/{N} docs → IDF = {idf_score:.4f}")
    
print("\n'china' is rarer than 'supplier', so it gets a HIGHER IDF — it's more informative!")

## Step 2: TF Saturation — Diminishing returns for repeated words

**TF_component = (TF * (k1 + 1)) / (TF + k1 * (1 - b + b * dl/avgdl))**

Where:
- k1 = 1.5 (saturation parameter)
- b = 0.75 (length normalization)
- dl = document length
- avgdl = average document length

The key insight: going from TF=0 to TF=1 is a BIG jump.
Going from TF=5 to TF=6 barely matters. This prevents keyword stuffing.

In [ ]:
k1 = 1.5
b = 0.75

doc_lengths = [len(doc) for doc in tokenized]
avgdl = sum(doc_lengths) / len(doc_lengths)

print(f"Doc lengths: {doc_lengths}")
print(f"Average doc length: {avgdl:.1f}")
print()

# Show TF saturation curve
print("TF saturation (how much each additional occurrence matters):")
dl = avgdl  # assume average length doc
for tf in range(0, 8):
    if tf == 0:
        score = 0
    else:
        score = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
    bar = '█' * int(score * 10)
    print(f"  TF={tf}: {score:.4f} {bar}")

## Step 3: Full BM25 Score

**score(q, d) = Σ IDF(t) × TF_component(t, d)**

For each query term, multiply its IDF by its TF component, then sum.

In [ ]:
def bm25_score(query_terms, doc_tokens, tokenized_corpus, k1=1.5, b=0.75):
    dl = len(doc_tokens)
    avgdl = sum(len(d) for d in tokenized_corpus) / len(tokenized_corpus)
    score = 0
    breakdown = []
    
    for term in query_terms:
        tf = doc_tokens.count(term)
        idf_val = idf(term, tokenized_corpus)
        
        if tf > 0:
            tf_component = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
        else:
            tf_component = 0
        
        term_score = idf_val * tf_component
        score += term_score
        breakdown.append(f"  '{term}': IDF={idf_val:.4f} × TF_comp={tf_component:.4f} = {term_score:.4f}")
    
    return score, breakdown

# Score each document
print(f"Query: '{query}'\n")
for i, doc_tokens in enumerate(tokenized):
    score, breakdown = bm25_score(query_terms, doc_tokens, tokenized)
    print(f"Doc {i}: \"{corpus[i]}\"")
    print(f"  BM25 Score = {score:.4f}")
    for line in breakdown:
        print(line)
    print()

## Key Takeaways

1. **IDF matters more than TF** — rare words are more informative
2. **TF saturates** — the 5th occurrence of a word barely changes the score
3. **Long documents get penalized** (controlled by `b`) — prevents bias toward verbose text
4. **BM25 fails on synonyms** — 'vendors' and 'suppliers' are different tokens, zero match

This is why we need Semantic Search (Technique 2) — it understands meaning, not just keywords.